# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**
Train readable → stronger classifiers, rank by predicted decline probability, and compare
against the Week-4 CTR-gap baseline on the **same client-holdout test set** and **precision@K**.

> Skills: `training-honest-models` + `flyrank/flyrank-data`.

## 1. Method choice and why

| Choice | Why it fits Lane 2 |
|---|---|
| **Question shape** | "Which pages first?" → ranking by a score, evaluated with **precision@K**. |
| **Label (proxy)** | `is_declining_label` from `trend_direction == down` — same proxy the Week-4 notebook used for receipts. Honest limit: snapshot-derived, not a future-window outcome. |
| **Logistic Regression** | Readable linear baseline among *learned* models; forces scaled features. |
| **Random Forest** | Stronger non-linear model that still gives feature importances; common next step after LR. |
| **Not starting with GBDT** | Complexity only if RF clearly beats LR *and* the baseline; we report all three. |

**Week-4 baseline to beat (frozen):** `impressions_90d >= 500` AND `0 < avg_position <= 20` AND `ctr < 0.5`,
scored as `log1p(impressions_90d)` among firings — reason `low_ctr_visible_page`.

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
        os.chdir("..")

CSV = Path("data/raw/content_refresh_anonymized.csv")
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUT_DIR / "w05_model_vs_baseline.json"

df = pd.read_csv(CSV)
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print("Working dir:", os.getcwd())
print(f"Rows: {len(df):,} | clients: {df['client_id'].nunique()} | declining rate: {df['is_declining_label'].mean():.3f}")
print("Methods: LogisticRegression + RandomForest vs Week-4 CTR-gap baseline")
print(f"sklearn/pandas/numpy seeds: RANDOM_STATE={RANDOM_STATE}")

Working dir: C:\Users\Windows 11\Flyrank_Ml_internship
Rows: 30,000 | clients: 32 | declining rate: 0.542
Methods: LogisticRegression + RandomForest vs Week-4 CTR-gap baseline
sklearn/pandas/numpy seeds: RANDOM_STATE=42


## 2. Split design

**Client holdout (grouped by `client_id`).** Pages from the same client share templates, seasons,
and update habits — a random row split leaks that structure into the test set and flatters the model.
Holding out ~20% of *clients* asks: does this ranking idea transfer to clients the model never saw?

**Features:** observable 90d / age / CTR / position signals.  
**Excluded from X:** `trend_direction`, `trend_pct`, `is_declining_label`, and the last/prev-30d
windows that define the trend proxy (label-adjacent).

In [2]:
# --- features (no trend / no last-prev 30d windows) ---
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["log_sessions_90d"] = np.log1p(df["sessions_90d"].clip(lower=0))
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["word_count_filled"] = df["word_count"].fillna(0)

NUMERIC_FEATURES = [
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "has_position",
    "engagement_rate",
    "scroll_rate",
    "word_count_filled",
    "has_word_count",
]

X_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].astype(int)
clients = df["client_id"].fillna("unknown").astype(str)

# --- client holdout ---
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2

print(f"Split: client_holdout | train clients={len(unique_clients) - n_test_clients} "
      f"test clients={n_test_clients}")
print(f"train rows={len(train_idx):,} (declining={y.iloc[train_idx].mean():.3f}) | "
      f"test rows={len(test_idx):,} (declining={y.iloc[test_idx].mean():.3f})")
print(f"Feature count: {len(NUMERIC_FEATURES)}")
print("Excluded from X: trend_direction, trend_pct, is_declining_label, *_last_30d, *_prev_30d")

Split: client_holdout | train clients=26 test clients=6


train rows=27,675 (declining=0.555) | test rows=2,325 (declining=0.391)
Feature count: 14
Excluded from X: trend_direction, trend_pct, is_declining_label, *_last_30d, *_prev_30d


## 3. Train + compare vs my baseline

Same test rows for everyone. Metric: **precision@10** and **precision@50** on the declining
proxy (plus ROC AUC / AP as secondary). Baseline score recomputed here so the table is one run.

In [3]:
def precision_at_k(scores: np.ndarray, labels: np.ndarray, k: int) -> float:
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(labels)[order[:k]]
    return float(top.mean()) if len(top) else 0.0


def week4_baseline_scores(frame: pd.DataFrame) -> np.ndarray:
    """Frozen Week-4 rule: low CTR on visible pages."""
    gate = (
        (frame["impressions_90d"] >= 500)
        & (frame["avg_position"] > 0)
        & (frame["avg_position"] <= 20)
        & (frame["ctr"] < 0.5)
    ).astype(float)
    return np.log1p(frame["impressions_90d"].clip(lower=0).to_numpy()) * gate.to_numpy()


X_tr, X_te = X_num.iloc[train_idx], X_num.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
test_frame = df.iloc[test_idx].reset_index(drop=True)

base_scores = week4_baseline_scores(test_frame)
base_rate = float(y_te.mean())

lr = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)
lr.fit(X_tr, y_tr)
lr_proba = lr.predict_proba(X_te)[:, 1]

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_tr, y_tr)
rf_proba = rf.predict_proba(X_te)[:, 1]

rows = []
for name, scores in [
    ("week4_ctr_gap_baseline", base_scores),
    ("logistic_regression", lr_proba),
    ("random_forest", rf_proba),
]:
    rows.append(
        {
            "method": name,
            "precision@10": round(precision_at_k(scores, y_te, 10), 4),
            "precision@50": round(precision_at_k(scores, y_te, 50), 4),
            "roc_auc": round(float(roc_auc_score(y_te, scores)), 4),
            "avg_precision": round(float(average_precision_score(y_te, scores)), 4),
        }
    )

comparison = pd.DataFrame(rows)
print(f"Test base rate (declining): {base_rate:.3f}")
print(f"Test n={len(y_te):,} | split=client_holdout | RANDOM_STATE={RANDOM_STATE}")
print("\nModel vs baseline (same test set):")
print(comparison.to_string(index=False))

best_p50 = comparison.sort_values("precision@50", ascending=False).iloc[0]
print(
    f"\nBest precision@50: {best_p50['method']} = {best_p50['precision@50']:.3f} "
    f"(baseline={comparison.loc[comparison['method']=='week4_ctr_gap_baseline','precision@50'].iloc[0]:.3f})"
)

Test base rate (declining): 0.391
Test n=2,325 | split=client_holdout | RANDOM_STATE=42

Model vs baseline (same test set):
                method  precision@10  precision@50  roc_auc  avg_precision
week4_ctr_gap_baseline           0.1          0.30   0.5572         0.4097
   logistic_regression           0.6          0.54   0.7145         0.5501
         random_forest           0.9          0.90   0.7641         0.6684

Best precision@50: random_forest = 0.900 (baseline=0.300)


## 4. Errors and interpretation

What the model leans on, where the top of the RF queue is wrong, and three concrete hard cases.

In [4]:
imp = pd.Series(rf.feature_importances_, index=NUMERIC_FEATURES).sort_values(ascending=False)
print("Random Forest — top features (impurity importance):")
print(imp.head(8).round(4).to_string())
print(
    "\nSanity: days_with_impressions / log traffic / position / age showing up is plausible "
    "for a decline-proxy ranker. A near-1.0 single feature would be a leakage smell — not seen here."
)

# Error slice: top-20 RF picks on the test set
te = test_frame.copy()
te["y"] = y_te
te["rf_proba"] = rf_proba
te["baseline_score"] = base_scores
top20 = te.sort_values("rf_proba", ascending=False).head(20)
print("\nRF top-20 on test: declining rate =", round(top20["y"].mean(), 3))
print("False positives in RF top-20 (y=0):", int((top20["y"] == 0).sum()))

fp = top20.loc[top20["y"] == 0].head(3)
fn_pool = te.loc[te["y"] == 1].sort_values("rf_proba").head(3)

print("\n--- 3 false positives at the top (model ranked high, label=0) ---")
for i, (_, r) in enumerate(fp.iterrows(), 1):
    print(
        f"{i}. {r['content_id']}: rf_proba={r['rf_proba']:.3f}, "
        f"imp={int(r['impressions_90d'])}, pos={r['avg_position']}, ctr={r['ctr']:.3f}, "
        f"days_since_update={int(r['days_since_last_update'])}"
    )
    print(
        "   Hard because: high demand + middling CTR/position can look 'at risk' even when "
        "the 30d trend proxy did not mark down — branded or stable pages."
    )

print("\n--- 3 false negatives (label=1 but model scored low) ---")
for i, (_, r) in enumerate(fn_pool.iterrows(), 1):
    print(
        f"{i}. {r['content_id']}: rf_proba={r['rf_proba']:.3f}, "
        f"imp={int(r['impressions_90d'])}, pos={r['avg_position']}, ctr={r['ctr']:.3f}"
    )
    print(
        "   Hard because: low-volume declines barely move the RF score — the model (and editors) "
        "correctly deprioritize pages with little demand, even if the proxy says 'down'."
    )

print(
    "\nReading the errors: misses concentrate on low-impression downs; false alarms concentrate "
    "on high-impression pages that are not currently labeled declining. For decision-support that "
    "is often acceptable — editor time should track demand — but it is not proof a refresh recovers traffic."
)

metrics = {
    "lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "test_rows": int(len(y_te)),
    "test_clients": int(n_test_clients),
    "base_rate_test": round(base_rate, 4),
    "features": NUMERIC_FEATURES,
    "comparison": comparison.to_dict(orient="records"),
    "top_rf_features": imp.head(8).round(4).to_dict(),
    "claim": (
        "Model scores are decision-support ranks on a snapshot decline proxy; "
        "not causal evidence that a refresh recovers traffic."
    ),
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"\nWrote {METRICS_PATH}")

Random Forest — top features (impurity importance):
avg_position             0.1566
log_impressions_90d      0.1456
content_age_days         0.1209
days_with_impressions    0.1080
word_count_filled        0.0886
scroll_rate              0.0635
log_sessions_90d         0.0602
ctr                      0.0595

Sanity: days_with_impressions / log traffic / position / age showing up is plausible for a decline-proxy ranker. A near-1.0 single feature would be a leakage smell — not seen here.



RF top-20 on test: declining rate = 0.95
False positives in RF top-20 (y=0): 1

--- 3 false positives at the top (model ranked high, label=0) ---
1. content_a1dd3f309e08: rf_proba=0.897, imp=6250, pos=13.3, ctr=0.110, days_since_update=20
   Hard because: high demand + middling CTR/position can look 'at risk' even when the 30d trend proxy did not mark down — branded or stable pages.

--- 3 false negatives (label=1 but model scored low) ---
1. content_28b4223f4e5f: rf_proba=0.020, imp=1, pos=0.0, ctr=0.000
   Hard because: low-volume declines barely move the RF score — the model (and editors) correctly deprioritize pages with little demand, even if the proxy says 'down'.
2. content_34b14c00f80c: rf_proba=0.061, imp=3, pos=0.0, ctr=0.000
   Hard because: low-volume declines barely move the RF score — the model (and editors) correctly deprioritize pages with little demand, even if the proxy says 'down'.
3. content_2cfc7a1fc728: rf_proba=0.102, imp=3, pos=5.0, ctr=0.000
   Hard because: l

## Self-check

- [x] Method choice explained (LR + RF for ranking / precision@K)
- [x] Client-holdout split; trend / last-prev-30d excluded from features
- [x] Comparison table vs Week-4 baseline on the **same** test set and metric
- [x] Feature importances + concrete FP/FN notes
- [x] Careful language: observed / decision-support — no causal refresh claims
- [x] Notebook runs top to bottom
- [x] Committed under `work/notebooks/` (+ metrics JSON)